# Fine-tune SAM-Road com o dataset local (SpaceNet 3)

Rota A do plano v2 (`training/finetune_local.md`). Pré-requisitos no seu Google Drive, pasta `samroad_finetune/`:
- `spacenet_samroad.zip` (conteúdo da pasta `TP_RP_2026_1\data\`)
- `spacenet_vitb_256_e10.ckpt`
- `sam_vit_b_01ec64.pth`

Runtime → Change runtime type → **T4 GPU**. Tempo esperado: 3–8 h para 8 épocas.

In [ ]:
!nvidia-smi -L
!git clone https://github.com/htcr/sam_road.git
%cd sam_road
!pip -q install lightning segment-anything-py opencv-python-headless scikit-image wandb addict
import torch; print('cuda:', torch.cuda.is_available())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/samroad_finetune'
!ls -la {DRIVE}

In [ ]:
# Celula de INSPECAO: descobre o layout de dados que o dataset.py espera
# antes de mover qualquer arquivo (autoverificacao).
!grep -n "rgb.png\|RGB_1.0\|spacenet" dataset.py | head -30

In [ ]:
# Ajuste DEST conforme a saida da celula anterior (padrao do repo original:
# ./spacenet/RGB_1.0_meter/). Os arquivos do zip ja estao no formato
# {AOI}__rgb.png / __gt.png / __gt_graph_dense.p esperado.
import os
DEST = 'spacenet/RGB_1.0_meter'
os.makedirs(DEST, exist_ok=True)
!unzip -q {DRIVE}/spacenet_samroad.zip -d {DEST}
print(len(os.listdir(DEST)), 'arquivos em', DEST)

os.makedirs('sam_ckpts', exist_ok=True)
!cp {DRIVE}/sam_vit_b_01ec64.pth sam_ckpts/
!cp {DRIVE}/spacenet_vitb_256_e10.ckpt ./

In [ ]:
# (Opcional) split por cidade: treinar Vegas+Xangai+Paris, validar Cartum.
# O repo gera/usa um data split json; se existir, filtre aqui. Caso o split
# seja gerado em runtime pelo dataset.py, pule esta celula.
!ls *.json spacenet/*.json 2>/dev/null || echo 'sem split json visivel — dataset.py decide'

In [ ]:
cfg = '''DATASET: 'spacenet'
NO_SAM: False
SAM_VERSION: 'vit_b'
SAM_CKPT_PATH: 'sam_ckpts/sam_vit_b_01ec64.pth'
PATCH_SIZE: 256
BATCH_SIZE: 32
DATA_WORKER_NUM: 2
TRAIN_EPOCHS: 8
BASE_LR: 0.0001
FREEZE_ENCODER: False
ENCODER_LR_FACTOR: 0.1
ENCODER_LORA: False
FOCAL_LOSS: True
USE_SAM_DECODER: False
TOPO_SAMPLE_NUM: 128
TOPONET_VERSION: 'normal'
INFER_BATCH_SIZE: 64
SAMPLE_MARGIN: 0
INFER_PATCHES_PER_EDGE: 16
ITSC_THRESHOLD: 0.195
ROAD_THRESHOLD: 0.341
TOPO_THRESHOLD: 0.705
ITSC_NMS_RADIUS: 8
ROAD_NMS_RADIUS: 16
NEIGHBOR_RADIUS: 64
MAX_NEIGHBOR_QUERIES: 16
'''
with open('config/finetune_spacenet_local.yaml', 'w') as f:
    f.write(cfg)
print(cfg)

In [ ]:
# (Opcional, recomendado) patch clDice no model.py — conectividade na origem.
# O arquivo cldice_loss.py e copiado automaticamente do seu Drive.
!cp {DRIVE}/cldice_loss.py .
import re, pathlib
src = pathlib.Path('model.py').read_text()
anchor = 'mask_loss = self.mask_criterion(mask_logits, gt_masks)'
patch = anchor + '''
        # --- clDice (plano v2): preserva conectividade do esqueleto ---
        try:
            from cldice_loss import soft_cldice_loss
            _road_pred = torch.sigmoid(mask_logits[..., 1])
            _road_gt = gt_masks[..., 1].float()
            mask_loss = mask_loss + 0.3 * soft_cldice_loss(_road_pred, _road_gt)
        except Exception as _e:
            print('clDice off:', _e)
'''
if anchor in src and 'soft_cldice_loss' not in src:
    pathlib.Path('model.py').write_text(src.replace(anchor, patch, 1))
    print('clDice aplicado (verifique a ordem dos canais de gt_masks!)')
else:
    print('Anchor nao encontrado ou ja aplicado — revise manualmente model.py')

In [ ]:
# Treino: driver proprio (equivalente ao train.py sem wandb), inicializando
# do checkpoint publicado — e isso que "agrega o dataset aos pesos".
import torch
import lightning.pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint
from torch.utils.data import DataLoader
from utils import load_config
from dataset import SatMapDataset, graph_collate_fn
from model import SAMRoad

config = load_config('config/finetune_spacenet_local.yaml')
net = SAMRoad(config)

ckpt = torch.load('spacenet_vitb_256_e10.ckpt', map_location='cpu')
missing, unexpected = net.load_state_dict(ckpt['state_dict'], strict=False)
print('init do ckpt publicado | faltando:', len(missing), '| extras:', len(unexpected))

train_ds = SatMapDataset(config, is_train=True)
val_ds = SatMapDataset(config, is_train=False)
train_loader = DataLoader(train_ds, batch_size=config.BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, collate_fn=graph_collate_fn)
val_loader = DataLoader(val_ds, batch_size=config.BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True, collate_fn=graph_collate_fn)

# checkpoints salvos DIRETO no Drive (sobrevivem a desconexao do Colab);
# mantem apenas os 2 melhores por val_loss para nao estourar a cota do Drive
ckpt_cb = ModelCheckpoint(dirpath=f'{DRIVE}/finetune_out', monitor='val_loss',
                          mode='min', save_top_k=2, every_n_epochs=1)
trainer = pl.Trainer(max_epochs=config.TRAIN_EPOCHS, precision=16,
                     callbacks=[ckpt_cb], default_root_dir='finetune_out')
trainer.fit(net, train_dataloaders=train_loader, val_dataloaders=val_loader)

In [ ]:
# Os checkpoints ja estao no Drive (a celula de treino salva la a cada epoca)
!ls -lh {DRIVE}/finetune_out/
print('Pronto. Baixe o melhor ckpt do Drive para TP_RP_2026_1/weights/finetuned_spacenet_local.ckpt')